In [ ]:
#Library Imports
import gymnasium as gym
from gymnasium import spaces
import spaces
import numpy as np
import random

In [ ]:
class DroneRescueEnv(gym.Env):
    def __init__(self):
        super(DroneRescueEnv, self).__init__()
        
        # Grid size
        self.grid_size = 6
        self.max_battery = 10
        self.maximum_steps = 75
        self.windprob = 0.3 # 30% chance of wind affecting movement
        # Define action space: Up, Down, Left, Right, Hover
        self.action_space = spaces.Discrete(5)
        
        # Observation space: (row, col, battery, rescued_targets_status)
        self.observation_space = spaces.Tuple((
            spaces.Discrete(self.grid_size),          # row position
            spaces.Discrete(self.grid_size),          # column position
            spaces.Discrete(self.max_battery+1),      # battery level
            spaces.MultiBinary(3)                     # rescue status (3 targets)
        ))

        

#   Symbol     Meaning 
#     S     Start position 
#     F     Free/Safe cell 
#     D     Dangerous zone 
#     R     Rescue target 
#     C     Charging station 
#     W     Wind zone 
#     X     Blocked cell / obstacle 
        # Grid configuration
        self.grid = np.array([
            ['S','F','F','D','F','F'],
            ['F','X','W','F','R','F'],
            ['F','F','D','F','F','W'],
            ['C','F','F','X','F','R'],
            ['F','F','D','F','W','F'],
            ['F','R','F','C','X','F']
        ])
        
        # Rescue target positions 
        self.rescue_targets = [(1,4), (3,5), (5,1)]
        self.start_pos = (0,0)
        self.reset()
    
    def reset(self):
        self.pos = self.start_pos
        self.battery = self.max_battery
        self.rescued = [0,0,0] 
        return (self.pos[0], self.pos[1], self.battery, np.array(self.rescued))
    

    #Action mapping: 0=Up, 1=Down, 2=Left, 3=Right, 4=Hover
    def step(self, action):
        r, c = self.pos
        # If current cell is wind zone, apply stochastic transition
        if self.grid[r,c] == 'W' and action in [0,1,2,3]:
            if random.random() < self.wind_prob:
                action = random.choice([0,1,2,3])  

        # Movement logic with boundary checks
        nr, nc = r, c
        if action == 0 and r > 0: nr -= 1                    # Up
        elif action == 1 and r < self.grid_size-1: nr += 1   # Down
        elif action == 2 and c > 0: nc -= 1                  # Left
        elif action == 3 and c < self.grid_size-1: nc += 1   # Right
        # Hover = stay
        
        # Check blocked cells
        if self.grid[nr,nc] == 'X':
            nr, nc = r, c  # stay in place
        
        self.pos = (nr,nc)
        self.battery -= 1
        self.steps += 1

        reward = -1  # move cost
        done = False
        
        cell = self.grid[nr,nc]
        if cell == 'R' and (nr,nc) not in self.rescued:
            for idx, target in enumerate(self.rescue_targets):
                if (nr,nc) == target and self.rescued[idx] == 0:
                    reward += 20
                    self.rescued[idx] = 1
        elif cell == 'C':
            reward += 5
            self.battery = self.max_battery
        elif cell == 'D':
            reward -= 10
        
        if self.battery <= 0:
            reward -= 20
            done = True
        if all(self.rescued):
            done = True
        if self.steps >= self.max_steps:
            print("Maximum steps crossed. Resetting environment.")
            done = True
        
        return (nr,nc,self.battery,np.array(self.rescued)), reward, done, {}


({'image': array([[[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0]],

       [[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0]],

       [[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0]],

       [[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]],

       [[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]],

       [[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]],

       [[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [8, 1, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]]], dtype=uint8), 'direction': 0, 'mission': 'ge

c:\Users\Mouli\AppData\Local\Programs\Python\Python311\Lib\site-packages\pygame\sysfont.py:494: UserWarning: The system font 'freesansbold.ttf' couldn't be found. Did you mean: 'freesansbold', 'freesans'? Verify your font name input. Using the default font instead.
  warnings.warn(


: 